# Pipeline v9 — residual dan modulasi (seluruh utterance)
Acuan alur: **glottal v9 saja**. Panduan residual-modulasi menentukan domain bukti.
Dataset dan split diimpor persis dari manifest run glottal v9. Kode, cache, model, dan hasil milik proyek ini.

**Cara kerja di Colab (dirancang ulang 23 Nov 2026).** Semua proses berat berjalan di disk lokal runtime
(`/content/work`): audio disalin sekali dari zip/folder dataset, fitur dan model ditulis di sana. Drive hanya
menerima *cadangan tulis-sekali* di `backup/`: shard fitur (CSV + tar frame, ±500 utterance per shard), seed
model yang sudah dibekukan, dan paket hasil akhir. Tidak ada file besar yang ditulis ulang di Drive.

Jalankan T00–T06 berurutan. Bila runtime terputus, jalankan ulang dari T00: audio di-*stage* ulang (±15–25 menit),
lalu T04/T05 memulihkan shard dan seed dari `backup/` dan hanya mengerjakan sisanya.


In [1]:
# T00 — lokasi proyek dan sumber dataset. Ubah hanya blok ini.
from pathlib import Path
import os, sys, subprocess, shutil, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/College/speech-deepfake-residual-modulation-new')
    GLOTTAL_ROOT = Path('/content/drive/MyDrive/College/speech-deepfake-glottal')
    WORK_ROOT = Path('/content/work')            # disk lokal runtime: semua proses berat di sini
else:
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/residual_modulation_v9.yaml').exists())
    GLOTTAL_ROOT = PROJECT_ROOT.parent/'speech-deepfake-glottal'
    WORK_ROOT = PROJECT_ROOT/'results/_work'
DATA_ROOT = GLOTTAL_ROOT/'data'
SOURCE_RUN = GLOTTAL_ROOT/'results/forensic_whole_utterance_v9/fe17e2b856dd5d6c'
PROFILE = 'full'       # full = pilihan v9 persis; smoke = 1 utterance/sel, subset eksplisit
N_JOBS = os.cpu_count() or 2                     # tidak memengaruhi identitas cache
BACKUP = PROJECT_ROOT/'backup'/PROFILE            # satu-satunya tujuan tulis di Drive
CODE_ROOT = PROJECT_ROOT
RUNNER = CODE_ROOT/'scripts/run_pipeline_v9.py'
assert RUNNER.is_file(), f'Salin src/, scripts/, configs/, requirements.txt ke {CODE_ROOT}'
assert PROJECT_ROOT.resolve() != GLOTTAL_ROOT.resolve(), 'Output wajib berada pada proyek residual-modulasi.'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(WORK_ROOT).free / 1e9
print(f'Kode: {PROJECT_ROOT}\nData: {DATA_ROOT}\nKerja lokal: {WORK_ROOT} ({free_gb:.0f} GB kosong)\nCadangan: {BACKUP}\nN_JOBS={N_JOBS}')
assert free_gb > 30, 'Disk lokal < 30 GB; gunakan runtime baru.'


Mounted at /content/drive
Kode: /content/drive/MyDrive/College/speech-deepfake-residual-modulation-new
Data: /content/drive/MyDrive/College/speech-deepfake-glottal/data
Kerja lokal: /content/work (116 GB kosong)
Cadangan: /content/drive/MyDrive/College/speech-deepfake-residual-modulation-new/backup/full
N_JOBS=2


In [2]:
# T00b — cek Drive akun Colab SEBELUM bekerja berjam-jam (Colab saja).
# File yang ditulis Colab memakai kuota akun yang login di Colab, walau foldernya milik akun lain.
# Uji ini mengunggah 1 MB lalu menghapusnya; kuota penuh langsung terdeteksi (storageQuotaExceeded).
if IN_COLAB:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaInMemoryUpload
    auth.authenticate_user()
    api = build('drive', 'v3')
    me = api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
    try:
        test = api.files().create(body={'name': '.colab_quota_test'}, media_body=MediaInMemoryUpload(os.urandom(1 << 20)),
                                  fields='id').execute()
        api.files().delete(fileId=test['id']).execute()
    except HttpError as error:
        raise SystemExit(f'Drive akun Colab ({me}) menolak file baru: {error}.\n'
                         'Kosongkan kuota akun ini (termasuk Sampah) atau login Colab dengan akun pemilik folder.')
    shortcut = '.shortcut-targets-by-id' in str(PROJECT_ROOT.resolve())
    print(f'Akun Colab: {me} — unggah uji OK.' + (' Folder proyek adalah shortcut milik akun lain; cadangan tetap '
          'memakai kuota akun Colab (±8–10 GB untuk run penuh).' if shortcut else ''))


Akun Colab: fawwasaliy11@gmail.com — unggah uji OK.


In [3]:
# T01 — environment
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_ROOT/'requirements.txt')], check=True)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
def run(*args):
    subprocess.run([sys.executable, str(RUNNER), *map(str, args)], check=True, cwd=CODE_ROOT)
run('--help')


## T02 — manifest yang sama dengan glottal v9
Importer hanya membaca `manifests/*.csv` dari run v9 terpilih (ID, label, dataset, split dipertahankan).
Manifest dibuat di disk lokal dan jumlah barisnya diperiksa; salinannya disimpan ke `backup/inputs`.


In [4]:
import pandas as pd
import yaml
from src.pipeline_v9.common import write_csv, write_json, file_hash, mirror_tree
MANIFESTS = WORK_ROOT/'manifests'
run('import-glottal-v9', '--source-run', SOURCE_RUN, '--data-root', DATA_ROOT, '--out', MANIFESTS)
manifest = pd.read_csv(MANIFESTS/'manifest.csv', keep_default_na=False)
expected = sum(v['rows'] for v in json.loads((MANIFESTS/'source_v9.json').read_text())['manifests'].values())
assert len(manifest) == expected, f'Manifest {len(manifest)} baris, seharusnya {expected}.'
def apply_profile(cfg):
    if PROFILE == 'smoke':
        cfg.update(seeds=[2026], bootstrap_draws=100, permutation_repeats=3)
        cfg['gmm'].update(n_components=2, max_frames=20000)
    elif PROFILE != 'full':
        raise ValueError('PROFILE harus full atau smoke')
    return cfg
settings = apply_profile(yaml.safe_load((CODE_ROOT/'configs/residual_modulation_v9.yaml').read_text()))
if PROFILE == 'smoke':
    manifest = pd.concat([g.sample(1, random_state=2026) for _, g in manifest.groupby(
        ['dataset', 'split', 'speaker_id', 'label', 'attack_id'])], ignore_index=True)
WORK = WORK_ROOT/PROFILE
INPUTS = WORK/'inputs'
MANIFEST, CONFIG = INPUTS/'input_manifest.csv', INPUTS/'config.yaml'
write_csv(MANIFEST, manifest)
CONFIG.write_text(yaml.safe_dump(settings, sort_keys=False))
shutil.copyfile(MANIFESTS/'source_v9.json', INPUTS/'source_v9.json')
write_json(INPUTS/'selection.json', {'profile': PROFILE, 'source_run': str(SOURCE_RUN),
    'source_manifest_sha256': file_hash(MANIFESTS/'manifest.csv'), 'n': len(manifest)})
mirror_tree(INPUTS, BACKUP/'inputs')
print(manifest.groupby(['dataset', 'split', 'label']).size())
print('Total:', len(manifest))


dataset          split     label
asvspoof2019_la  dev       0          500
                           1         1500
                 eval      0         1675
                           1        15600
                 train     0         2580
                           1        22800
asvspoof2021_df  external  0          187
                           1          625
mlaad_tiny       external  0          953
                           1          953
dtype: int64
Total: 47373


## T03 — stage audio ke disk lokal + audit
LA 2019 dibaca dari `LA.zip` yang disalin ke disk lokal (satu bacaan berurutan 7,6 GB jauh lebih cepat daripada
44 ribu file kecil di Drive). DF 2021 dibaca langsung dari `DF.zip` (hanya 812 file). MLAAD disalin per file.
SHA-256 dihitung dari byte yang ditulis, lalu dicek kebocoran speaker/audio. Tidak ada yang ditulis ke folder glottal.


In [5]:
STAGE = WORK_ROOT/'stage'
run('stage', '--manifest', MANIFEST, '--out', STAGE,
    '--zip', f"asvspoof2019_la={DATA_ROOT/'external/asvspoof2019_la/LA.zip'}",
    '--zip', f"asvspoof2021_df={DATA_ROOT/'external/asvspoof2021_df/DF.zip'}",
    *(['--copy-zip', 'asvspoof2019_la'] if PROFILE == 'full' else []), '--jobs', 16)
STAGED = STAGE/'staged_manifest.csv'
staged = pd.read_csv(STAGED, keep_default_na=False)
assert len(staged) == len(manifest) and staged.audio_status.eq('ready').all()
for name in ['staged_manifest.csv', 'staged_manifest.json']:
    shutil.copyfile(STAGE/name, INPUTS/name)
mirror_tree(INPUTS, BACKUP/'inputs')
print('Audio siap:', len(staged), '| disk kosong:', f'{shutil.disk_usage(WORK_ROOT).free / 1e9:.0f} GB')


Audio siap: 47373 | disk kosong: 95 GB


In [6]:
# T04 — ekstraksi seluruh utterance. Shard tulis-sekali dicadangkan ke Drive dan dipulihkan saat dilanjutkan.
FEATURES = WORK/'features'
run('extract', '--manifest', STAGED, '--config', CONFIG, '--out', FEATURES,
    '--jobs', N_JOBS, '--chunk', 500, '--backup', BACKUP/'features')
display(pd.read_csv(FEATURES/'coverage.csv'))


,dataset,split,label,n,extracted,coverage
0,asvspoof2019_la,dev,0,500,500,1.0
1,asvspoof2019_la,dev,1,1500,1500,1.0
2,asvspoof2019_la,eval,0,1675,1675,1.0
3,asvspoof2019_la,eval,1,15600,15600,1.0
4,asvspoof2019_la,train,0,2580,2580,1.0
5,asvspoof2019_la,train,1,22800,22800,1.0
6,asvspoof2021_df,external,0,187,187,1.0
7,asvspoof2021_df,external,1,625,625,1.0
8,mlaad_tiny,external,0,953,953,1.0
9,mlaad_tiny,external,1,953,953,1.0


In [7]:
# T05 — train/dev, bekukan semua seed, baru evaluasi target dan laporan G1–G7.
# Seed yang sudah beku langsung dicadangkan; paket hasil disalin ke Drive setelah selesai.
RUNS, CHECKPOINTS = WORK/'runs', WORK/'checkpoints'
# Sesi baru hanya untuk T07/T08? Isi RUN_ID run yang sudah selesai di Drive (mis. '1d5ba7b2b9562c28'):
# paket hasil dan seed beku disalin dari backup tanpa training/evaluasi ulang. T04 tetap perlu dijalankan.
RESTORE_RUN_ID = '1d5ba7b2b9562c28'
if RESTORE_RUN_ID:
    mirror_tree(BACKUP/'runs'/RESTORE_RUN_ID, RUNS/RESTORE_RUN_ID)
    mirror_tree(BACKUP/'checkpoints'/RESTORE_RUN_ID, CHECKPOINTS/RESTORE_RUN_ID)
    RUN_ID = RESTORE_RUN_ID
else:
    run('run', '--features', FEATURES, '--config', CONFIG, '--out', RUNS, '--checkpoints', CHECKPOINTS, '--backup', BACKUP)
    RUN_ID = (RUNS/'LATEST_RUN').read_text().strip()
RUN_DIR = RUNS/RUN_ID
print('Run:', RUN_ID)


Run: 1d5ba7b2b9562c28


In [8]:
# T06 — ringkasan; SD antarseed berbeda dari CI bootstrap speaker.
assert (RUN_DIR/'complete.json').is_file(), f'Run belum lengkap: {RUN_DIR}'
display(pd.read_csv(RUN_DIR/'seed_summary.csv'))
display(pd.read_csv(RUN_DIR/'evidence_consistency.csv'))
# Kontrol shortcut: selisih AUC berpasangan (bootstrap speaker, seed pertama). CI tidak memuat 0 = berbeda.
ci = pd.read_csv(RUN_DIR/'bootstrap_intervals.csv')
if 'comparison' in ci:
    controls = ci[ci.comparison.notna() & ci.status.eq('paired_difference') & ci.metric.eq('auc')
                  & ci.seed.eq(settings['seeds'][0])]
    display(controls[['scenario', 'comparison', 'median', 'low', 'high']].round(3))
# Analisis shortcut hening/durasi (tanpa training ulang): tabel di RUN_DIR/analysis, disalin ke Drive.
run('analyze', '--run', RUN_DIR)
mirror_tree(RUN_DIR/'analysis', BACKUP/'runs'/RUN_ID/'analysis')
strat = pd.read_csv(RUN_DIR/'analysis'/'stratified_auc_summary.csv')
display(strat[strat.split.ne('dev')].pivot_table(index='system', columns=['dataset', 'split'],
        values=['pooled_auc_mean', 'auc_within_silence_seconds_mean']).round(3))
print('Hasil lokal:', RUN_DIR)
print('Hasil di Drive:', BACKUP/'runs'/RUN_ID)


,scenario,system,auc_mean,auc_std,eer_mean,eer_std,f1_fake_mean,f1_fake_std,balanced_accuracy_mean,balanced_accuracy_std,fpr_mean,fpr_std,fnr_mean,fnr_std
0,asvspoof2019_la__dev,LFCC__gmm,0.984168,0.000423,0.062000,0.003464,0.957793,0.002406,0.938000,0.003464,0.062000,0.003464,0.062000,0.003464
1,asvspoof2019_la__dev,LFCC__hgb,0.980858,0.000090,0.071889,0.002009,0.950938,0.001415,0.928111,0.002009,0.072000,0.002000,0.071778,0.002037
2,asvspoof2019_la__dev,LFCC__lr,0.975581,0.000000,0.078000,0.000000,0.946612,0.000000,0.922000,0.000000,0.078000,0.000000,0.078000,0.000000
3,asvspoof2019_la__dev,LFCC__rf,0.984560,0.000503,0.062667,0.001155,0.957331,0.000803,0.937333,0.001155,0.062667,0.001155,0.062667,0.001155
4,asvspoof2019_la__dev,LFCC__svm_rbf,0.993956,0.000000,0.036000,0.000000,0.975709,0.000000,0.964000,0.000000,0.036000,0.000000,0.036000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,mlaad_tiny__external,Residual__hgb,0.426595,0.014468,0.554390,0.014552,0.506144,0.010162,0.434942,0.007922,0.709339,0.003783,0.420776,0.015670
204,mlaad_tiny__external,Residual__lr,0.466374,0.000000,0.517314,0.000000,0.546684,0.000000,0.437041,0.000000,0.804827,0.000000,0.321091,0.000000
205,mlaad_tiny__external,Residual__rf,0.451374,0.004146,0.547744,0.004809,0.449846,0.009761,0.452606,0.004241,0.542497,0.006881,0.552291,0.014247
206,mlaad_tiny__external,Residual__svm_rbf,0.476201,0.000000,0.518363,0.000000,0.514257,0.000000,0.490556,0.000000,0.558237,0.000000,0.460651,0.000000


,seed,scenario,baseline_system,evidence_system,n,spearman_rho,spearman_p,agreement,cohen_kappa,n_both_correct,n_baseline_correct_evidence_wrong,n_baseline_wrong_evidence_correct,n_both_wrong
0,2026,asvspoof2019_la__dev,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,2000,0.360809,1.517414e-62,0.815500,0.559858,1630,318,51,1
1,2026,asvspoof2019_la__eval,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,17275,0.112085,2.050402e-49,0.515195,0.035815,4561,5836,2539,4339
2,2026,asvspoof2021_df__external,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,812,0.100721,4.066014e-03,0.562808,0.062935,347,229,126,110
3,2026,mlaad_tiny__external,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,1906,0.093349,4.471478e-05,0.619098,0.066810,268,123,603,912
4,2027,asvspoof2019_la__dev,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,2000,0.366652,1.119858e-64,0.819000,0.567474,1637,311,51,1
5,2027,asvspoof2019_la__eval,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,17275,0.103532,2.211135e-42,0.518495,0.042261,4613,5784,2534,4344
6,2027,asvspoof2021_df__external,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,812,0.103224,3.231945e-03,0.568966,0.078810,351,225,125,111
7,2027,mlaad_tiny__external,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,1906,0.093812,4.095901e-05,0.616999,0.053766,268,123,607,908
8,2028,asvspoof2019_la__dev,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,2000,0.366660,1.111996e-64,0.821000,0.571768,1641,307,51,1
9,2028,asvspoof2019_la__eval,MFCC_LFCC__svm_rbf,Residual_Modulation__rf,17275,0.116953,1.137551e-53,0.524110,0.053213,4710,5687,2534,4344


,scenario,comparison,median,low,high
36,asvspoof2019_la__dev,QC_plus_Residual_Modulation vs QC_only (lr),0.158,0.115,0.207
45,asvspoof2019_la__dev,QC_plus_Residual_Modulation vs QC_only (hgb),0.161,0.130,0.191
54,asvspoof2019_la__dev,QC_plus_MFCC_LFCC vs QC_only (lr),0.234,0.175,0.285
63,asvspoof2019_la__dev,QC_plus_MFCC_LFCC vs QC_only (hgb),0.194,0.163,0.230
72,asvspoof2019_la__dev,Residual_Modulation vs MFCC_LFCC (lr),-0.114,-0.139,-0.091
81,asvspoof2019_la__dev,Residual_Modulation vs MFCC_LFCC (hgb),-0.077,-0.101,-0.058
120,asvspoof2019_la__eval,QC_plus_Residual_Modulation vs QC_only (lr),-0.008,-0.019,0.003
129,asvspoof2019_la__eval,QC_plus_Residual_Modulation vs QC_only (hgb),0.017,0.006,0.029
138,asvspoof2019_la__eval,QC_plus_MFCC_LFCC vs QC_only (lr),0.013,-0.004,0.029
147,asvspoof2019_la__eval,QC_plus_MFCC_LFCC vs QC_only (hgb),-0.040,-0.058,-0.025


auc_within_silence_seconds_mean  \
dataset                                              asvspoof2019_la   
split                                                           eval   
system                                                                 
LFCC__gmm                                                      0.883   
LFCC__hgb                                                      0.796   
LFCC__lr                                                       0.882   
LFCC__rf                                                       0.780   
LFCC__svm_rbf                                                  0.845   
LFCC__xgb                                                      0.815   
MFCC_LFCC__hgb                                                 0.849   
MFCC_LFCC__lr                                                  0.899   
MFCC_LFCC__rf                                                  0.834   
MFCC_LFCC__svm_rbf                                             0.887   
MFCC_LFCC__xgb                                                 0.865   
MFCC__gmm                                                      0.801   
MFCC__hgb                                                      0.783   
MFCC__lr                                                       0.764   
MFCC__rf                                                       0.800   
MFCC__svm_rbf                                                  0.759   
MFCC__xgb                                                      0.791   
Modulation__hgb                                                0.595   
Modulation__lr                                                 0.575   
Modulation__rf                                                 0.588   
Modulation__svm_rbf                                            0.565   
Modulation__xgb                                                0.600   
QC_only__hgb                                                   0.684   
QC_only__lr                                                    0.644   
QC_only__rf                                                    0.708   
QC_only__svm_rbf                                               0.683   
QC_only__xgb                                                   0.694   
QC_plus_MFCC_LFCC__hgb                                         0.852   
QC_plus_MFCC_LFCC__lr                                          0.899   
QC_plus_MFCC_LFCC__rf                                          0.840   
QC_plus_MFCC_LFCC__svm_rbf                                     0.890   
QC_plus_MFCC_LFCC__xgb                                         0.869   
QC_plus_Residual_Modulation__hgb                               0.756   
QC_plus_Residual_Modulation__lr                                0.680   
QC_plus_Residual_Modulation__rf                                0.734   
QC_plus_Residual_Modulation__svm_rbf                           0.782   
QC_plus_Residual_Modulation__xgb                               0.766   
Residual_Modulation__hgb                                       0.713   
Residual_Modulation__lr                                        0.648   
Residual_Modulation__rf                                        0.701   
Residual_Modulation__svm_rbf                                   0.691   
Residual_Modulation__xgb                                       0.694   
Residual__hgb                                                  0.675   
Residual__lr                                                   0.578   
Residual__rf                                                   0.633   
Residual__svm_rbf                                              0.585   
Residual__xgb                                                  0.615   

                                                                 \
dataset                              asvspoof2021_df mlaad_tiny   
split                                       external   external   
system                                                            
LFCC__gmm                                      0.691      0.276   
LFCC__hgb                    

Hasil lokal: /content/work/full/runs/1d5ba7b2b9562c28
Hasil di Drive: /content/drive/MyDrive/College/speech-deepfake-residual-modulation-new/backup/full/runs/1d5ba7b2b9562c28


## T07 — sensitivitas setelah model dibekukan (opsional)
Mengubah satu faktor per varian (orde LPC 12/20, trim 25/45 dB/tanpa trim, noise SNR 30/20/10, MP3 128/64/32 kbps)
pada subset target yang tetap: `per-cell` utterance per dataset × split × speaker × label × serangan.
`per-cell 1` = 1.294 utterance × 12 varian (±1,5 jam pada 2 core); `per-cell 2` = 2.336 utterance (±2,5–3 jam).
Model seed pertama dan threshold tetap; GMM tidak disertakan. Ini menguji sensitivitas pengukuran, bukan tuning target.
Shard fitur tiap varian dan tabel hasil dicadangkan ke `backup/<PROFILE>/stress/<RUN_ID>/`; menjalankan ulang melanjutkan.


In [9]:
RUN_STRESS = False     # ubah ke True untuk menjalankan
STRESS_PER_CELL = 1    # 1 ≈ separuh waktu dari 2
if RUN_STRESS:
    run('stress', '--run', RUN_DIR, '--features', FEATURES, '--checkpoints', CHECKPOINTS,
        '--per-cell', STRESS_PER_CELL, '--jobs', N_JOBS, '--backup', BACKUP/'stress'/RUN_ID)
    display(papakd.read_csv(RUN_DIR/'stress'/'variant_status.csv'))
    metrics = pd.read_csv(RUN_DIR/'stress'/'metrics.csv')
    display(metrics.pivot_table(index='variant', columns=['dataset', 'system'], values='auc').round(3))


## T08 — ablation pita modulasi (opsional, ±1–2 jam training, tanpa ekstraksi ulang)
Melatih ulang dengan `configs/residual_modulation_v9_band_ablation.yaml`: tiap pita modulasi **sendirian**, keempat pita
tanpa centroid/entropy/variabilitas, dan fitur bentuk tanpa pita. Energi pita berjumlah ≈1, jadi "tanpa satu pita"
tidak informatif (bisa direkonstruksi dari tiga sisanya). Fitur memakai cache T04 yang sama; GMM dimatikan.
Eksperimen development terpisah; jangan memilih konfigurasi dari hasil eksternal.


In [10]:
RUN_ABLATION = True   # ubah ke True untuk menjalankan
if RUN_ABLATION:
    ABLATION_CONFIG = INPUTS/'config_band_ablation.yaml'
    ABLATION_CONFIG.write_text(yaml.safe_dump(apply_profile(yaml.safe_load(
        (CODE_ROOT/'configs/residual_modulation_v9_band_ablation.yaml').read_text())), sort_keys=False))
    mirror_tree(INPUTS, BACKUP/'inputs')
    ABLATION_RUNS = WORK/'runs_band_ablation'
    run('run', '--features', FEATURES, '--config', ABLATION_CONFIG, '--out', ABLATION_RUNS,
        '--checkpoints', WORK/'checkpoints_band_ablation', '--backup', BACKUP/'band_ablation')
    ABLATION_DIR = ABLATION_RUNS/(ABLATION_RUNS/'LATEST_RUN').read_text().strip()
    summary = pd.read_csv(ABLATION_DIR/'seed_summary.csv')
    display(summary[summary.system.str.startswith('Modulation')].pivot_table(
        index='system', columns='scenario', values='auc_mean').round(3))
    print('Hasil ablation di Drive:', BACKUP/'band_ablation'/'runs'/ABLATION_DIR.name)


scenario,asvspoof2019_la__dev,asvspoof2019_la__eval,asvspoof2021_df__external,mlaad_tiny__external
system,,,,
Modulation__hgb,0.679,0.631,0.529,0.487
Modulation__lr,0.649,0.638,0.502,0.528
Modulation__rf,0.646,0.607,0.514,0.484
Modulation__svm_rbf,0.662,0.621,0.531,0.422
Modulation__xgb,0.671,0.623,0.513,0.496
Modulation_band_0p5_2_only__hgb,0.551,0.543,0.528,0.527
Modulation_band_0p5_2_only__lr,0.543,0.551,0.544,0.535
Modulation_band_0p5_2_only__rf,0.528,0.507,0.439,0.508
Modulation_band_0p5_2_only__svm_rbf,0.532,0.548,0.531,0.533


Hasil ablation di Drive: /content/drive/MyDrive/College/speech-deepfake-residual-modulation-new/backup/full/band_ablation/runs/52760a671e6853be
